# 1. Similarity (default)

- retreever = store.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 3}
)

- docs = retriever.invoke("kargom ne zaman gelir?")

    - En yakın K döküman (Benzer sonuçlar tekrar edebilir)

# 2. MMR (Maximim Marginal Relevance)

- Relevance + Diversity dengesi

- retriever = store.as_retriever(
    search = "mmr",
    search_kwargs = (
        "k": 3,
        "fetch_k: 10,
        "lambda_mull": 0.5
    )
)

- docs = retriever.invoke("iade ve değişim nasıl yapılıyor?")
1. lambda_mull = 0 - en çeşitli
2. lamda_mull = 1 -> en relevant

# 3. Threshold Filtering

- retriever = store.as_retriver(
    search_type = "similarity_score_threshold",
    search_kwargs = {
        "score_threshold": 0.3,
        "k": 5,
        "filter": {"kategori": "Kargo & Teslimat"},
    } 
)

- docs = retriever.invoke("kargo icreti ne kadar?")
    - Sadece 0.3+ score + sadece bu kategori

# 4. Hybrid Search

- Dense(embedding) + Sparse(BM25)

- from langchain_community.retrievers import BM2Retriever
- from langchain.retrievers import EmsambleRetriever

- bm25 = BM25Retriever.from_documents(doc)
- dense = store.as_retriever()

- hybrid = EnsembleRetriever(
    retrievers = [bm25, dense],
    weights = [0.4,0.6]
)
- docs = hybrid.invoke("kapıda ödeme var mı
)

# 5. Re-ranking

1. İlk retrieval: 20 döküman getir
2. Re-rank: daha güçlü model ile sırala, en iyi 5'i al

- from langchain_cohere import ChoreRerank
- from langchain.retrievers import ContexttualCompressionRetriever

- compressor = CohoreRerank(model = "rerank-v3.5")

- rerank = ContexttualCompressionRetriever(
    base_compressor = compressor,
    base_retriever = base_retriever,
)
- docs = raranker.invoke("hesabımı nasıl silerim")

# İleri Teknikler (2026)

- Bunları şimdi kurman gerekmez - sadece isimlerini bil, işine gelince derinleşirsin. Retrieval'ın bir sonu yok.

- Agentic RAG - sabit pipeline yerine LLM aramayı kendi yönetir: ne zaman/neyi getireceğine karar verir, kötüyse sorguyu yeniden yazıp tekrar dener. (2026'nın yükselen konusu.)

- Parent-document -- kücük parçayla ARA (isabet), LLM'e büyük parçayı VER(bağlam), Chunk-boyutu ikilemi çözer.

- Contextual Retrieval -- her chunk'a kısa bir "nereye ait" bağlamı ekleyip öyle embed et -> isabet artar

# K Seçimi

-  K     -----           Kullanım
- 2-3    -----         FAQ, kesin cevap
- 3-5    -----         Default, çoğu durumda
- 5-10   -----         Uzun cevap, sentez
- 10+    -----         Gürültü riski      

# Özet 

- Similarity: basit, default
- MMR: çeşitlilik gerekirse
- Hybrid (BM25 + dense): production standart
- Re-rank: en yüksek kalite

# Retrieval stratejileri -- Sepetix Destek Asistanı

- Vektör DB hazır. Şimdi en iyi chunk'ları getirme (retrieval) stratejilerine bakıyoruz -- "RAG" hatalarının ~%70'i retrieval'dadır": yanlış belge gelirse cevap da yanlış olur. Embedding OpenAI ile (.env -> OPENAI_API_KEY)  

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_chroma import Chroma
import os

# 0. İndeksi kur(ml1 + ml2 özeti)

In [ ]:
docs = DirectoryLoader("sirket-bilgi-bankasi/", glob = "**/*.md", loader_cls = TextLoader)
chunks = RecursiveCharacterTextSplitter(chunk_size = 400, chunk_overlap = 60).split()
KATEGORİ = {
    "kargo-teslimat": "Kargo & Teslimat", "iade-degisim": "İade & Değişim",
    "siparis-odeme": "Siparis & Ödeme", "uyelik-sadakat": "Üyelik & Sadakat",
    "sss": "Sıkça Sorulan Sorular"
}
for ch in chunks:
    ad = os.path.basename(ch.metadata["source"]).replace(".md","")
    ch.metadata["kategori"] = KATEGORİ.get(ad,ad)
load_dotenv()
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
vectordb = Chroma.from_documents(chunks, embeddings = embeddings) 
print(len(chunks), "chunk indekslendi")

# 1. Retriever - temel arayüz

- as_retriever vektör DB'yi bir 'getirici''ye çevirir, k kaç chunk döneceğidir.

In [ ]:
retriever = vectordb.as_retriever(search_kwargs = {"k":2})
for d in retriever.invoke("siparişimi nasıl iptal ederim?"):
    print(f"[{d.metadata["kategori"]}] {d.page_content[:60].strip()}...")

# 2. MMR -- çeşitlilik (tekrarı önle)


- Similarity en yakın 3'ü getirir ama 3'de aynı şeyi söyleyebilir. MMR alaka + çeşitlilik dengeler.

In [ ]:
mmr = vectordb.as_retriever(search_type =  "mmr", search_kwargs = {"k":3, "fetch_k": 8, "lambda_mult": 0.5})
q = "ödeme ve kargo ile ilgili bilgiler"
print("Similarity", [d.metadata["kategori"] for d in vectordb.similarity_search()])
print("MMR (çeşitli):", [d.metadata["kategori"] for d in mmr.invoke(q)])

# 3. Skor eşiği -- Kalite kontrol

In [ ]:
th = vectordb.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs = {
        "score_threshold": 0.1,
        "k": 4
    })
print(len(th.invoke("iade süresi nedir?")), "sonuç eşiği geçti")

# 4. Metadata filtresi